# overview
plot the influence of number of niches in domain list; 

 BASS reuslts included



In [ ]:
from src.paths import dataset_dir, dataset_file, dataset_root, repository_root
import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scipy
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, completeness_score

from src.utils import *
import src.prompt as prompt
from src.data_loader import load_spatial_data_csv



In [ ]:
# test  domain list with zeroshot
test_domain_list_results = pd.read_csv("examples/results/test_domain_list_all_metrics.csv")
test_domain_list_results['num_niches'] = test_domain_list_results['niche_test_id'].astype(str).str.len()



# original zeroshot results
original_results_df = pd.read_csv('examples/results/zeroshot_all_metrics_all_replicates.csv')
original_results_df = original_results_df[original_results_df['data_type'] == 'starmap']
original_results_df = original_results_df[original_results_df['model_name'] == 'gpt4o_mini']
original_results_df['num_niches'] = 4

# BASS results
bass_results_df = pd.read_csv('examples/results/BASS_K345_all_metrics.csv')

In [ ]:
test_domain_list_results

In [ ]:

bass_results_df

In [ ]:
plot_columns = ['data_name', 'model_name', 'num_niches', 'NMI', 'ARI', 'HOM', 'COM']
# Combine the dataframes using only the columns in plot_columns
dfs_to_concat = [
    bass_results_df, 
    original_results_df,
    test_domain_list_results
]

# Ensure the necessary columns exist and have the same types
for df in dfs_to_concat:
    if 'num_niches' in df:
        df['num_niches'] = df['num_niches'].astype(int)

# Only select relevant columns, dropping others
combined_df = pd.concat(
    [df[plot_columns] for df in dfs_to_concat],
    axis=0,
    ignore_index=True
)

# Show shape as a sanity check
print('Combined dataframe shape:', combined_df.shape)
combined_df.head()


In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['pdf.fonttype'] = 42
import seaborn as sns
from scipy import stats

def plot_metric_by_niches(combined_df, metric_name, figsize=(10, 6), title=None):
    """
    Plot a line chart showing the relationship between num_niches and a specified metric,
    with separate lines for each model_name and standard error bars.
    
    Parameters:
    -----------
    combined_df : pd.DataFrame
        DataFrame containing columns: 'num_niches', 'model_name', and the specified metric
    metric_name : str
        Name of the metric to plot on y-axis ('NMI', 'ARI', 'COM', etc.)
    figsize : tuple, optional
        Figure size (width, height). Default is (10, 6)
    title : str, optional
        Plot title. If None, uses default title based on metric_name
    
    Returns:
    --------
    fig, ax : matplotlib figure and axis objects
    """
    
    # Validate inputs
    if metric_name not in combined_df.columns:
        raise ValueError(f"Metric '{metric_name}' not found in DataFrame columns: {list(combined_df.columns)}")
    
    # Create figure and axis
    fig, ax = plt.subplots(figsize=figsize)
    
    # Set color palette
    unique_models = combined_df['model_name'].unique()
    colors = ['#1f77b4', '#f7b6d2']# sns.color_palette("husl", len(unique_models))
    
    # Plot for each model
    for i, model in enumerate(unique_models):
        model_data = combined_df[combined_df['model_name'] == model]
        
        # Group by num_niches and calculate mean and standard error
        grouped = model_data.groupby('num_niches')[metric_name].agg(['mean', 'std', 'count']).reset_index()
        
        # Calculate standard error
        grouped['se'] = grouped['std'] / np.sqrt(grouped['count'])
        
        # Plot line with error bars
        ax.errorbar(
            grouped['num_niches'], 
            grouped['mean'], 
            yerr=grouped['se'],
            marker='o', 
            linewidth=2, 
            markersize=7,
            capsize=5,
            capthick=2,
            color=colors[i],
            label=model
        )
    
    # Customize plot
    ax.set_xlabel('Number of Niches', fontsize=12, fontweight='bold')
    ax.set_ylabel(metric_name, fontsize=12, fontweight='bold')
    
    if title is None:
        title = f'{metric_name} vs Number of Niches by Model'
    ax.set_title(title, fontsize=14, fontweight='bold')
    
    # Add legend
    ax.legend( title_fontsize=11, fontsize=10, loc='best')
    
    # Add grid for better readability
    # ax.grid(False, alpha=0.3)
    
    # Set x-axis to show integer ticks only
    ax.set_xticks(sorted(combined_df['num_niches'].unique()))
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Improve layout
    plt.tight_layout()
    
    return fig, ax


In [ ]:
# Example usage of the plotting function

# Plot NMI
fig1, ax1 = plot_metric_by_niches(combined_df, 'NMI', figsize=(3.5, 6))
plt.savefig("figures/NMI_vs_num_niches_by_model.pdf")
plt.show()

# Plot ARI
fig2, ax2 = plot_metric_by_niches(combined_df, 'ARI')
plt.show()

# Plot COM (Completeness)
fig3, ax3 = plot_metric_by_niches(combined_df, 'COM')
plt.show()


In [ ]:
# Let's examine the data structure to understand data_name distribution
print("Unique data_name values:")
print(combined_df['data_name'].unique())
print(f"\nNumber of unique data_name values: {combined_df['data_name'].nunique()}")

print("\nData distribution by data_name and model_name:")
print(combined_df.groupby(['data_name', 'model_name']).size().unstack(fill_value=0))

print("\nData distribution by data_name and num_niches:")
print(combined_df.groupby(['data_name', 'num_niches']).size().unstack(fill_value=0))


In [ ]:
def plot_metric_by_niches_subplots(combined_df, metric_name, figsize=None, suptitle=None, ncols=None):
    """
    Plot subplots showing the relationship between num_niches and a specified metric,
    with separate subplots for each data_name and separate lines for each model_name.
    All subplots share the same y-axis limits for comparability.
    Handles cases where models have insufficient data for error bars.
    
    Parameters:
    -----------
    combined_df : pd.DataFrame
        DataFrame containing columns: 'data_name', 'num_niches', 'model_name', and the specified metric
    metric_name : str
        Name of the metric to plot on y-axis ('NMI', 'ARI', 'COM', etc.)
    figsize : tuple, optional
        Figure size (width, height). If None, automatically calculated based on number of subplots
    suptitle : str, optional
        Overall figure title. If None, uses default title based on metric_name
    ncols : int, optional
        Number of columns for subplot layout. If None, automatically determined
    
    Returns:
    --------
    fig, axes : matplotlib figure and axis objects
    """
    # Validate inputs
    if metric_name not in combined_df.columns:
        raise ValueError(f"Metric '{metric_name}' not found in DataFrame columns: {list(combined_df.columns)}")
    
    # Compute y-axis limits for all data (shared for all subplots)
    y_min = combined_df[metric_name].min()
    y_max = combined_df[metric_name].max()
    y_pad = 0.05 * (y_max - y_min) if (y_max - y_min) > 0 else 0.05  # minimal padding
    
    # "Tighten" to common metrics scale (mainly for scores in [0,1])
    lower_lim = max(0.0, y_min - y_pad)
    upper_lim = min(1.0, y_max + y_pad) if y_max <= 1.0 else y_max + y_pad

    # Get unique data names and models
    unique_data_names = sorted(combined_df['data_name'].unique())
    unique_models = combined_df['model_name'].unique()
    n_data = len(unique_data_names)
    
    # Determine subplot layout
    if ncols is None:
        ncols = min(3, n_data)  # Max 3 columns for readability
    nrows = (n_data + ncols - 1) // ncols  # Ceiling division
    
    # Calculate figure size if not provided
    if figsize is None:
        figsize = (5 * ncols, 4 * nrows)
    
    # Create subplots
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    
    # Handle case where there's only one subplot
    if n_data == 1:
        axes = [axes]
    elif nrows == 1:
        axes = axes.flatten() if ncols > 1 else [axes]
    else:
        axes = axes.flatten()
    
    # Set color palette
    colors = sns.color_palette("husl", len(unique_models))
    model_colors = dict(zip(unique_models, colors))
    
    # Plot for each data_name
    for i, data_name in enumerate(unique_data_names):
        ax = axes[i]
        data_subset = combined_df[combined_df['data_name'] == data_name]
        
        # Plot for each model within this data_name
        for model in unique_models:
            model_data = data_subset[data_subset['model_name'] == model]
            
            if len(model_data) == 0:
                continue  # Skip if no data for this model in this dataset
            
            # Group by num_niches and calculate statistics
            grouped = model_data.groupby('num_niches')[metric_name].agg(['mean', 'std', 'count']).reset_index()
            
            # Handle cases with insufficient data for error bars
            # If count <= 1, we can't calculate meaningful standard error
            sufficient_data = grouped['count'] > 1
            
            if sufficient_data.any():
                # Calculate standard error only for groups with sufficient data
                grouped.loc[sufficient_data, 'se'] = (
                    grouped.loc[sufficient_data, 'std'] / 
                    np.sqrt(grouped.loc[sufficient_data, 'count'])
                )
                grouped.loc[~sufficient_data, 'se'] = 0  # No error bars for insufficient data
                
                # Plot with error bars where available
                error_bars = grouped['se'].values
                error_bars[~sufficient_data] = None  # No error bars for single points
                
                ax.errorbar(
                    grouped['num_niches'], 
                    grouped['mean'], 
                    yerr=error_bars,
                    marker='o', 
                    linewidth=2, 
                    markersize=6,
                    capsize=5,
                    capthick=2,
                    color=model_colors[model],
                    label=model,
                    alpha=0.8
                )
                
                # Add markers for points with insufficient data (single points)
                if (~sufficient_data).any():
                    insufficient_points = grouped[~sufficient_data]
                    ax.scatter(
                        insufficient_points['num_niches'], 
                        insufficient_points['mean'],
                        color=model_colors[model],
                        s=100,
                        marker='s',  # Square marker to distinguish from error bar points
                        alpha=0.6,
                        edgecolor='white',
                        linewidth=1
                    )
            else:
                # All points have insufficient data - just plot as scatter
                ax.scatter(
                    grouped['num_niches'], 
                    grouped['mean'],
                    color=model_colors[model],
                    s=100,
                    marker='s',
                    alpha=0.6,
                    label=model,
                    edgecolor='white',
                    linewidth=1
                )
        
        # Customize each subplot
        ax.set_xlabel('Number of Niches', fontsize=10, fontweight='bold')
        ax.set_ylabel(metric_name, fontsize=10, fontweight='bold')
        ax.set_title(f'{data_name}', fontsize=11, fontweight='bold')
        ax.grid(True, alpha=0.3)
        
        # Set x-axis to show integer ticks only
        data_niches = sorted(data_subset['num_niches'].unique())
        if data_niches:
            ax.set_xticks(data_niches)
        # Set y-axis to global limits (same for all subplots)
        ax.set_ylim(lower_lim, upper_lim)
        
        # Add legend to first subplot only
        if i == 0:
            ax.legend(title='Model Name', title_fontsize=9, fontsize=8, loc='best')
    
    # Hide extra subplots if any
    for j in range(n_data, len(axes)):
        axes[j].set_visible(False)
    
    # Set overall title
    if suptitle is None:
        suptitle = f'{metric_name} vs Number of Niches by Dataset and Model'
    fig.suptitle(suptitle, fontsize=14, fontweight='bold', y=0.98)
    
    # Improve layout
    plt.tight_layout()
    plt.subplots_adjust(top=0.88)  # Make room for suptitle
    
    # Add a note about square markers
    fig.text(0.02, 0.02, 'Note: Square markers indicate single data points (no error bars)', 
             fontsize=8, style='italic', alpha=0.7)
    
    return fig, axes


In [ ]:
# Example usage of the subplot function

# Plot NMI with subplots for each data_name
fig1, axes1 = plot_metric_by_niches_subplots(combined_df, 'NMI')
plt.show()

# Plot ARI with subplots
fig2, axes2 = plot_metric_by_niches_subplots(combined_df, 'ARI')
plt.show()

# Plot COM with custom parameters
fig3, axes3 = plot_metric_by_niches_subplots(
    combined_df, 
    'COM'
)
plt.show()
